# developer-agent experiment run
Runs `run_experiment.py` from our private bundle dataset. **Edit only the next cell**, then Save Version (Save & Run All).

In [ ]:
import time
SESSION_START = time.time()
RUNS = [1, 2, 3]             # one after another in this session; 0 = smoke test
SETUPS = ['D']              # setups to run: A, B, C, D; [] = A, B, C
TASKS = []                   # [] = all selected tasks (26); e.g. ['requests_7502'] for a test
RESUME = True                # continue from the bundle's resume/run<N>/ if it exists
MAX_HOURS = 6.0              # stop starting attempts after this; set to min(11, GPU quota left / 2), since 4x L4 counts double


In [ ]:
import glob, os, re, signal, subprocess, sys
script = glob.glob('/kaggle/input/**/developer-agent-bundle/run_experiment.py', recursive=True)
assert len(script) == 1, f'Bundle not found: {script}'
bundle = os.path.dirname(script[0])
print(open(os.path.join(bundle, 'BUNDLE_VERSION.txt')).read())
deadline = SESSION_START + MAX_HOURS * 3600
DEADLINE_EXIT = 3  # run_experiment.py: stopped at the deadline
SHOW = re.compile(r'^(\[\d\d:\d\d:\d\d\] |Timeout \(|Traceback|Fatal)')  # our log lines, crash reports


BOOT = next(int(l.split()[1]) for l in open('/proc/stat') if l.startswith('btime'))


def started_after_session(pid):
    """Only touch processes this notebook started (not Kaggle's own)."""
    fields = open(f'/proc/{pid}/stat').read().rsplit(')', 1)[1].split()
    return BOOT + int(fields[19]) / os.sysconf('SC_CLK_TCK') > SESSION_START


def gpu_pids():
    """Processes started by this notebook that hold a GPU open (vLLM server and its workers)."""
    pids = set()
    for fd_dir in glob.glob('/proc/[0-9]*/fd'):
        pid = int(fd_dir.split('/')[2])
        try:
            if pid == os.getpid() or not started_after_session(pid):
                continue
        except OSError:
            continue
        try:
            if any(os.readlink(os.path.join(fd_dir, f)).startswith('/dev/nvidia') for f in os.listdir(fd_dir)):
                pids.add(pid)
        except OSError:
            pass
    return pids


def gpus_free():
    """True if every GPU has < 1 GB in use; False if busy or nvidia-smi doesn't answer."""
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                             capture_output=True, text=True, timeout=30).stdout
        used = [int(x) for x in out.split()]
        print(f'GPU memory in use (MiB): {used}', flush=True)
        return bool(used) and max(used) < 1024
    except Exception as e:
        print(f'nvidia-smi failed: {e!r}', flush=True)
        return False


def clean_up(proc):
    """Kill the run script's whole process group, then anything still holding a GPU. Every step has a time limit."""
    try:
        os.killpg(proc.pid, signal.SIGKILL)
    except OSError:
        pass
    for attempt in range(6):  # up to ~3 min
        for pid in gpu_pids():
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
        time.sleep(30)
        if gpus_free():
            return True
    return False


def run_one(RUN):
    """One run, restarting the script after a hang. Returns True if the session should stop (deadline or trouble)."""
    cmd = [sys.executable, '-u', script[0], '--run', str(RUN), '--deadline', str(deadline)]
    cmd += ['--tasks', *TASKS] if TASKS else []
    cmd += ['--setups', *SETUPS] if SETUPS else []
    resume = os.path.join(bundle, 'resume', f'run{RUN}')
    if RESUME and os.path.isdir(resume):
        cmd += ['--resume-from', resume]
    # The script's output goes to a file, never to this notebook's output pipe: when Kaggle stops
    # draining that pipe, any write blocks forever. That froze run 1 twice (Oct 21, Oct 22).
    console_dir = f'/kaggle/working/results/run{RUN}'
    os.makedirs(console_dir, exist_ok=True)
    # The script exits if an attempt or startup hangs (hang guard). Clean up and restart; it skips
    # finished attempts, retries a hung one once, then records it as an error. The notebook stops
    # (and saves results) at the deadline, on a clean finish, or if the GPUs can't be freed.
    for session in range(6):
        remaining = deadline - time.time()
        if remaining < 15 * 60:
            print('Deadline reached; not restarting', flush=True)
            return True
        console = os.path.join(console_dir, f'console_{int(time.time())}.log')
        with open(console, 'w') as out:
            first = session == 0 and RUN == RUNS[0]  # install the harness once per Kaggle session
            proc = subprocess.Popen(cmd + ([] if first else ['--skip-install']), start_new_session=True,
                                    stdin=subprocess.DEVNULL, stdout=out, stderr=subprocess.STDOUT,
                                    env={**os.environ, 'PYTHONUNBUFFERED': '1'})
        code, offset, stop_at = None, 0, time.time() + remaining + 10 * 60  # backstop if the script ignores the deadline
        while code is None:
            try:
                code = proc.wait(timeout=60)
            except subprocess.TimeoutExpired:
                if time.time() > stop_at:
                    code = 'killed at deadline'
            with open(console, errors='replace') as f:  # show progress: only our short log lines
                f.seek(offset)
                new = f.read()
                offset = f.tell()
            for line in new.splitlines():
                if SHOW.match(line):
                    print(line[:200], flush=True)
        print(f'Run script ended: {code}', flush=True)
        freed = clean_up(proc)
        if not freed:
            print('GPUs still busy after cleanup; stopping so the results are saved', flush=True)
            return True
        if code in (0, DEADLINE_EXIT) or code == 'killed at deadline':
            return code != 0  # 0 = run finished: go on to the next run
    print('Gave up after 6 restarts; check the log', flush=True)
    return True


for RUN in RUNS:
    print(f'=== Run {RUN} ===', flush=True)
    if run_one(RUN):
        break


In [ ]:
# Zip results so they download as one file per run
import shutil
for RUN in RUNS:
    if os.path.isdir(f'/kaggle/working/results/run{RUN}'):
        print(shutil.make_archive(f'/kaggle/working/results_run{RUN}', 'zip', f'/kaggle/working/results/run{RUN}'))
